## Using the trained model to predict in our dataset

### First we import the packages and the dataset

In [17]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

# Make src/ importable: add the project root (parent of notebooks/) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Run from the project root so relative paths like "data/raw/..." resolve
os.chdir(project_root)
print(f"Working dir: {os.getcwd()}")

import pandas as pd

# Start by importing the test data
test_path = Path('data/raw/dataset_test.csv')
df_test = pd.read_csv(test_path)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Working dir: c:\Users\rcsrc\Documents\GitHub\predictive_modeling_and_mobility_forecasting


### Now we perform the datacleaning

In [18]:
from src.data_cleaning import clean_data

df_test = clean_data(dataset= df_test, is_train = False, categorize_station=True)

df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 537445 entries, 0 to 537444
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   id              537445 non-null  str           
 1   datetime        537445 non-null  datetime64[us]
 2   station_number  537445 non-null  category      
 3   name            537445 non-null  str           
 4   lat             537445 non-null  float64       
 5   lng             537445 non-null  float64       
 6   hour            537445 non-null  int32         
 7   minute          537445 non-null  int32         
 8   dayofweek       537445 non-null  int32         
 9   month           537445 non-null  int32         
 10  is_weekend      537445 non-null  int64         
 11  hour_sin        537445 non-null  float64       
 12  hour_cos        537445 non-null  float64       
 13  is_holiday      537445 non-null  int64         
dtypes: category(1), datetime64[us](1), float64(4), 

In [19]:
# Save IDs and do preprocessing (as we did before)

test_ids = df_test["id"].copy()      # save the ids

DROP_COLS = ["id", "datetime", "name"]#,"lat", "lng" 

df_test = df_test.drop(columns = DROP_COLS)


### Do the predictions

In [20]:
import joblib
import pandas as pd

# Load the trained model
model_path = Path('models/lightgbm.joblib')
model = joblib.load(model_path)

# Make predictions
predictions = model.predict(df_test)

# (Optional) For classifiers, get probability estimates
# probabilities = model.predict_proba(df_test)

ValueError: train and valid dataset categorical_feature do not match.

### Prepare the submission

In [10]:
submission = pd.DataFrame({"id": test_ids, "bikes": predictions})

# and now save the predictions
from pathlib import Path

out_path = Path('reports/predictions.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(out_path, index=False)